# Unidade IV — Mineração de Padrões

## Avaliação responsável de regras

**Carga estimada:** 2 horas  
**Pré-requisitos:** suporte, confiança, *lift*, *leverage* e *conviction*.

> **Pergunta norteadora:** como selecionar regras úteis sem ser enganado por uma única métrica ou atribuir causalidade a uma associação?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- reconhecer situações em que uma métrica isolada produz avaliação incompleta;
- interpretar conjuntamente suporte, confiança, *lift*, *leverage* e *conviction*;
- diferenciar ganho relativo, impacto absoluto, volume e violações da regra;
- incorporar estabilidade, utilidade, custos e redundância à seleção;
- comunicar associações sem transformá-las em conclusões causais.


In [1]:
import pandas as pd


## Regra forte não é sinônimo de regra útil

O [notebook 04.03](04_03_avaliacao_e_padroes_sequenciais.ipynb) apresentou as fórmulas e os cálculos das métricas. Aqui, o foco muda da pergunta “como calcular?” para “como decidir?”.

Uma análise defensável combina:

- **força descritiva:** o que as métricas informam sobre frequência e dependência;
- **volume:** quantas transações realmente sustentam a regra;
- **estabilidade:** se o resultado reaparece em outros períodos, segmentos ou amostras;
- **novidade e redundância:** se a regra acrescenta informação ou apenas repete outra;
- **acionabilidade:** se existe uma decisão concreta, legal e viável associada ao padrão;
- **custos e riscos:** consequências de agir quando a regra falha;
- **explicações alternativas:** promoções, disponibilidade, sazonalidade, exposição e segmentação.

Testar muitas regras aumenta a chance de encontrar coincidências. Um período de confirmação, reamostragem ou conjunto separado ajuda a reduzir conclusões oportunistas. Mesmo uma regra estável continua sendo observacional: estimar o efeito de uma intervenção exige desenho causal apropriado.


## Cinco cenários para comparar as métricas

Construiremos cinco cenários com $N=1.000$ transações. Cada cenário é definido pelas quatro regiões de uma tabela de contingência:

| Contagem | Significado |
|---|---|
| $n_{11}$ | transações com $A$ e $B$ |
| $n_{10}$ | transações com $A$ e sem $B$: violações de $A\rightarrow B$ |
| $n_{01}$ | transações sem $A$ e com $B$ |
| $n_{00}$ | transações sem $A$ e sem $B$ |

A partir dessas contagens, calcularemos conjuntamente:

$$\operatorname{sup}=P(A\cap B),\qquad
\operatorname{conf}=P(B\mid A),\qquad
\operatorname{lift}=\frac{P(B\mid A)}{P(B)},$$

$$\operatorname{leverage}=P(A\cap B)-P(A)P(B),\qquad
\operatorname{conviction}=\frac{P(\neg B)}{P(\neg B\mid A)}.$$

Os cenários foram construídos para isolar armadilhas de interpretação; não representam recomendações prontas.


In [2]:
casos_comparativos = pd.DataFrame([
    {"caso": "1. Confiança alta, mas independência", "n11": 180, "n10": 20, "n01": 720, "n00": 80},
    {"caso": "2. Regra perfeita, mas rara", "n11": 5, "n10": 0, "n01": 45, "n00": 950},
    {"caso": "3. Lift 2, baixo volume", "n11": 4, "n10": 16, "n01": 96, "n00": 884},
    {"caso": "4. Lift 2, alto volume", "n11": 320, "n10": 80, "n01": 80, "n00": 520},
    {"caso": "5. Associação negativa", "n11": 120, "n10": 280, "n01": 380, "n00": 220},
])

def calcular_cinco_medidas(linha):
    n = linha[["n11", "n10", "n01", "n00"]].sum()
    p_a = (linha["n11"] + linha["n10"]) / n
    p_b = (linha["n11"] + linha["n01"]) / n
    p_ab = linha["n11"] / n
    confidence = p_ab / p_a
    return pd.Series({
        "support": p_ab,
        "confidence": confidence,
        "lift": confidence / p_b,
        "leverage": p_ab - p_a * p_b,
        "conviction": (1 - p_b) / (1 - confidence)
        if confidence < 1 else float("inf"),
    })

medidas_casos = casos_comparativos.apply(calcular_cinco_medidas, axis=1)
medidas_casos.loc[medidas_casos["leverage"].abs() < 1e-12, "leverage"] = 0.0
tabela_casos = pd.concat([
    casos_comparativos[["caso", "n11", "n10"]], medidas_casos
], axis=1)

assert abs(medidas_casos.loc[0, "confidence"] - 0.9) < 1e-12
assert abs(medidas_casos.loc[0, "lift"] - 1) < 1e-12
assert abs(medidas_casos.loc[1, "confidence"] - 1) < 1e-12
assert abs(medidas_casos.loc[2, "lift"] - 2) < 1e-12
assert abs(medidas_casos.loc[3, "lift"] - 2) < 1e-12
assert medidas_casos.loc[4, "leverage"] < 0

tabela_casos.round(4)


,caso,n11,n10,support,confidence,lift,leverage,conviction
0,"1. Confiança alta, mas independência",180,20,0.180,0.9,1.0,0.0000,1.0000
1,"2. Regra perfeita, mas rara",5,0,0.005,1.0,20.0,0.0048,inf
2,"3. Lift 2, baixo volume",4,16,0.004,0.2,2.0,0.0020,1.1250
3,"4. Lift 2, alto volume",320,80,0.320,0.8,2.0,0.1600,3.0000
4,5. Associação negativa,120,280,0.120,0.3,0.6,-0.0800,0.7143


## Como avaliar cada caso

Na tabela, $n_{11}$ é a contagem conjunta e $n_{10}$ é a quantidade de violações da regra.

### Caso 1 — Confiança alta, mas nenhuma associação

A confiança de 0,90 parece forte: 90% das transações com $A$ possuem $B$. Entretanto, $B$ já aparece em 90% de toda a base. Por isso, *lift* = 1, *leverage* = 0 e *conviction* = 1. O antecedente não acrescenta informação sobre o consequente. **A confiança isolada produziria uma conclusão enganosa.**

### Caso 2 — Regra perfeita, mas apoiada por apenas cinco transações

A confiança é 1, o *lift* é 20 e a *conviction* é infinita, pois não há violação entre as cinco ocorrências de $A$. Porém, o suporte é somente 0,005 e o *leverage* é 0,00475: a regra envolve 0,5% da base e representa apenas 4,75 coocorrências acima da independência em 1.000 transações. Pode ser um nicho real, um caso valioso ou apenas uma coincidência instável; é indispensável examinar volume e validação fora da amostra.

### Casos 3 e 4 — Mesmo *lift*, impactos muito diferentes

As duas regras possuem *lift* = 2. No Caso 3, há apenas 4 coocorrências, suporte 0,004, *leverage* 0,002 e 16 violações; a confiança é 0,20 e a *conviction*, 1,125. No Caso 4, há 320 coocorrências, suporte 0,32, *leverage* 0,16, confiança 0,80 e *conviction* 3. **O mesmo ganho relativo não implica o mesmo alcance, impacto absoluto ou frequência de exceções.**

### Caso 5 — Coocorrência existente, mas associação negativa

$A$ e $B$ aparecem juntos em 120 transações, gerando suporte 0,12 e confiança 0,30. Como $B$ ocorre em 50% da base, esperaríamos 200 coocorrências sob independência. O *lift* é 0,60, o *leverage* é $-0{,}08$ — 80 coocorrências a menos que o esperado — e a *conviction* é aproximadamente 0,714. A presença de uma contagem conjunta relevante não garante associação positiva.

## Roteiro de leitura conjunta

| Pergunta | Medida que ajuda a responder | Cuidado principal |
|---|---|---|
| A regra envolve volume suficiente? | suporte e contagem $n_{11}$ | padrões raros podem ser instáveis |
| Quando $A$ ocorre, com que frequência $B$ ocorre? | confiança | pode apenas refletir um $B$ muito comum |
| Quanto a frequência condicional supera a taxa-base de $B$? | *lift* | ganho relativo alto pode ocorrer com volume mínimo |
| Quantas coocorrências excedem a independência em termos absolutos? | *leverage* | valores dependem das frequências marginais |
| Quão raras são as violações em relação ao esperado? | *conviction* | pode ser infinita em amostras pequenas sem violações |

Olhar todas as medidas **não significa exigir que todas sejam altas nem calcular uma média entre elas**. Significa usá-las como perguntas complementares e, depois, considerar estabilidade, custos, utilidade, redundância e explicações alternativas antes de agir.


> **U04-NB05-V01 — Verifique seu entendimento:** por que a confiança 0,90 do Caso 1 não torna a regra informativa? Cite as três métricas que revelam independência.

> **U04-NB05-E01 — Exercício:** compare os Casos 3 e 4. Calcule o excesso absoluto de coocorrências como $N\times\operatorname{leverage}$, compare as violações e explique por que o mesmo *lift* não justifica a mesma decisão.

> **U04-NB05-E02 — Atividade integradora:** escolha uma base transacional pequena; documente unidade, período e representação; minere regras com dois limiares; selecione no máximo cinco resultados usando as cinco métricas, estabilidade e utilidade; e produza uma recomendação com contagens, limitações e ressalva causal.


## Síntese

- Confiança alta pode apenas refletir um consequente muito comum.
- *Lift* alto ou *conviction* infinita não compensam automaticamente um volume mínimo.
- Regras com o mesmo *lift* podem ter impactos absolutos e taxas de violação muito diferentes.
- Nenhuma métrica basta sozinha; as cinco respondem a perguntas complementares.
- A decisão final também exige estabilidade, utilidade, custos e linguagem não causal.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 4.
